In [ ]:
import sagemaker
from datasets import load_dataset
import pandas as pd
from transformers import AutoTokenizer
import boto3
import os

sagemaker_session = sagemaker.Session()
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
#TODO - copy dataset to S3
local_dataset_type = "finance"

local_dataset_location = f"./data/{local_dataset_type}/"

!aws s3 cp --recursive {local_dataset_location} s3://{bucket_name}/datasets/modelcustomization/dpo/{local_dataset_type}/dpo_training.jsonl

dpo_dataset_s3_path = f"s3://{bucket_name}/datasets/modelcustomization/dpo/finance/"

In [ ]:
from sagemaker.config import load_sagemaker_config
configs = load_sagemaker_config()
from sagemaker.modules.train import ModelTrainer
from sagemaker.modules.configs import Compute, SourceCode, InputData, StoppingCondition, CheckpointConfig

env = {}
env["HF_token"] = "" #enter your HuggingFace token here for gated models
env["data_location"] = dpo_dataset_s3_path
env["training_recipe"] = "recipes/DPO-gemma-3-4b-it.yaml" #choose a recipe from the recipes folder

# MLFlow tracker
tracking_server_arn = ""
env["MLFLOW_TRACKING_ARN"] = tracking_server_arn

compute = Compute(
    instance_count=1,
    instance_type= "ml.p4de.24xlarge", # "ml.p3dn.24xlarge",
    volume_size_in_gb=500,
    keep_alive_period_in_seconds=3600,
)

image_uri = (
    #f"658645717510.dkr.ecr.{sagemaker_session.boto_session.region_name}.amazonaws.com/smdistributed-modelparallel:2.4.1-gpu-py311-cu121"
    f'763104351884.dkr.ecr.{sagemaker_session.boto_session.region_name}.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-ec2'
)

print(image_uri)

# checkpoint_s3_path = f"s3://{bucket_name}/dpo-checkpoints/checkpoints"
# print(checkpoint_s3_path)

job_prefix = f"dpo-gemma-3-4b-it"
print(job_prefix)

In [ ]:
hyperparameters = {
    #"dataset_path": "/opt/ml/input/data/dataset",
    #"model_dir": "/opt/ml/model",
}

source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="run_training_dpo.sh",
)


In [ ]:
model_trainer = ModelTrainer(
    training_image=image_uri,
    compute=compute,
    hyperparameters=hyperparameters,
    environment=env,
    source_code=source_code,
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=90000,
    ),
    # checkpoint_config=CheckpointConfig(
    #     s3_uri=f"{checkpoint_s3_path}/{job_prefix}",
    # ),
    base_job_name=job_prefix

)

In [ ]:
dpo_dataset_s3_path

In [ ]:
training_data = InputData(
    channel_name="training_dataset",
    data_source=dpo_dataset_s3_path,
)

In [ ]:
model_trainer.train(input_data_config=[training_data], wait=True)